# Chess Neural Network — Full Training Pipeline

Run each section in order.

1. Setup
2. Write source files
3. Download + process Lichess data
4. Supervised training (15 epochs)
5. RL self-play training (MCTS)
6. Download results


## 1. Setup

In [24]:
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU — connect to a GPU runtime for faster training')


GPU: NVIDIA A100-SXM4-80GB
Memory: 85.1 GB


In [25]:
!pip install -q python-chess zstandard tqdm requests


: 

In [26]:
import os
for d in ['/content/chess-nn/chess_nn',
          '/content/chess-nn/data/raw',
          '/content/chess-nn/data/processed',
          '/content/chess-nn/checkpoints_new',
          '/content/chess-nn/logs_new']:
    os.makedirs(d, exist_ok=True)
print('Directories created.')


Directories created.


: 

## 2. Write source files

In [27]:
%%writefile /content/chess-nn/config.py
"""Central configuration."""
import os
import torch

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

NUM_RESIDUAL_BLOCKS = 5
NUM_FILTERS         = 128
INPUT_PLANES        = 18
POLICY_OUTPUT_SIZE  = 4672

BATCH_SIZE     = 1024
LEARNING_RATE  = 0.001
WEIGHT_DECAY   = 1e-4
NUM_EPOCHS     = 15
GRADIENT_CLIP  = 1.0
VALUE_LOSS_WEIGHT = 0.5

MIN_RATING = 2000
MIN_MOVES  = 10
TRAIN_SPLIT = 0.90
VAL_SPLIT   = 0.05
TEST_SPLIT  = 0.05

RL_GAMES_PER_ITER  = 25
RL_SIMULATIONS     = 200
RL_CHUNK_SIZE      = 5
RL_HISTORY_FILES   = 5
RL_EPOCHS          = 5
RL_LR              = 1e-4
RL_EVAL_GAMES      = 20
RL_WIN_THRESHOLD   = 0.55

PROJECT_DIR        = "/content/chess-nn"
DATA_DIR           = os.path.join(PROJECT_DIR, "data")
RAW_DATA_DIR       = os.path.join(DATA_DIR, "raw")
PROCESSED_DATA_DIR = os.path.join(DATA_DIR, "processed")
CHECKPOINT_DIR     = os.path.join(PROJECT_DIR, "checkpoints_new")
LOG_DIR            = os.path.join(PROJECT_DIR, "logs_new")


Overwriting /content/chess-nn/config.py


: 

In [28]:
%%writefile /content/chess-nn/chess_nn/__init__.py


Overwriting /content/chess-nn/chess_nn/__init__.py


: 

In [29]:
%%writefile /content/chess-nn/chess_nn/board_encoding.py
"""
Board Encoding — converting a chess position into a tensor the neural network can read.

A chess board has 64 squares and 12 piece types (6 per color). We represent the board
as a stack of 18 binary "planes", each an 8x8 grid. Think of it like 18 transparent
sheets of paper, each one highlighting where a specific piece type sits.

Why tensors? Neural networks only understand numbers — this encoding translates
the game state into a 3D grid of 0s and 1s that a CNN can process.
"""

import numpy as np
import chess


# Map each piece type to its plane index (0-5 for white, 6-11 for black)
# chess.PAWN=1, chess.KNIGHT=2, ..., chess.KING=6
PIECE_TO_PLANE = {
    (chess.PAWN,   chess.WHITE): 0,
    (chess.KNIGHT, chess.WHITE): 1,
    (chess.BISHOP, chess.WHITE): 2,
    (chess.ROOK,   chess.WHITE): 3,
    (chess.QUEEN,  chess.WHITE): 4,
    (chess.KING,   chess.WHITE): 5,
    (chess.PAWN,   chess.BLACK): 6,
    (chess.KNIGHT, chess.BLACK): 7,
    (chess.BISHOP, chess.BLACK): 8,
    (chess.ROOK,   chess.BLACK): 9,
    (chess.QUEEN,  chess.BLACK): 10,
    (chess.KING,   chess.BLACK): 11,
}


def board_to_tensor(board: chess.Board) -> np.ndarray:
    """
    Convert a chess position to an (18, 8, 8) numpy array.

    The 18 planes are:
      0-5:  White pieces  (pawn, knight, bishop, rook, queen, king)
      6-11: Black pieces  (same order)
      12:   White can castle kingside  (all 1s or all 0s)
      13:   White can castle queenside
      14:   Black can castle kingside
      15:   Black can castle queenside
      16:   En passant target square  (1 on that square, 0 elsewhere)
      17:   Side to move  (all 1s = white to move, all 0s = black to move)

    Crucially, we always encode from the CURRENT PLAYER's perspective:
    if it's Black's turn, we flip the board so Black is always "at the bottom."
    This means the network only needs to learn one perspective, halving complexity.
    """
    planes = np.zeros((18, 8, 8), dtype=np.float32)
    flip = board.turn == chess.BLACK  # Flip board if it's Black's turn

    # --- Planes 0-11: Piece positions ---
    for square, piece in board.piece_map().items():
        row = chess.square_rank(square)  # 0 = rank 1 (bottom), 7 = rank 8 (top)
        col = chess.square_file(square)  # 0 = a-file, 7 = h-file

        if flip:
            row = 7 - row  # Flip vertically so current player is always "at bottom"

        plane_idx = PIECE_TO_PLANE[(piece.piece_type, piece.color)]

        # When flipped, also swap which planes represent "my pieces" vs "opponent's pieces"
        if flip:
            # Black's pieces (planes 6-11) become "my pieces" (planes 0-5) and vice versa
            if plane_idx < 6:
                plane_idx += 6
            else:
                plane_idx -= 6

        planes[plane_idx, row, col] = 1.0

    # --- Planes 12-15: Castling rights ---
    # These are binary flags — entire plane is 1 if right exists, 0 if not
    if not flip:
        planes[12] = float(board.has_kingside_castling_rights(chess.WHITE))
        planes[13] = float(board.has_queenside_castling_rights(chess.WHITE))
        planes[14] = float(board.has_kingside_castling_rights(chess.BLACK))
        planes[15] = float(board.has_queenside_castling_rights(chess.BLACK))
    else:
        # From Black's perspective, Black's castling rights are "my" rights (planes 12-13)
        planes[12] = float(board.has_kingside_castling_rights(chess.BLACK))
        planes[13] = float(board.has_queenside_castling_rights(chess.BLACK))
        planes[14] = float(board.has_kingside_castling_rights(chess.WHITE))
        planes[15] = float(board.has_queenside_castling_rights(chess.WHITE))

    # --- Plane 16: En passant target square ---
    if board.ep_square is not None:
        ep_row = chess.square_rank(board.ep_square)
        ep_col = chess.square_file(board.ep_square)
        if flip:
            ep_row = 7 - ep_row
        planes[16, ep_row, ep_col] = 1.0

    # --- Plane 17: Side to move ---
    # All 1s if white to move (from white's perspective), all 0s if black
    # Since we always flip to current player's view, this is always white-to-move = 1
    if board.turn == chess.WHITE:
        planes[17] = 1.0

    return planes


Overwriting /content/chess-nn/chess_nn/board_encoding.py


: 

In [30]:
%%writefile /content/chess-nn/chess_nn/move_encoding.py
"""
Move Encoding — converting chess moves into numbers the network can predict.

The policy head outputs a probability for each of 4672 possible "move slots."
We need a consistent mapping: every legal move maps to exactly one index,
and every index maps back to at most one legal move.

We use AlphaZero's encoding: 73 move planes × 64 source squares = 4672.
The 73 planes cover every possible move geometry a piece can make.
"""

import chess
import numpy as np

# --- Move plane layout (73 planes per source square) ---
#
# Planes 0-55: "Queen-style" moves (also covers pawn pushes/captures)
#   8 directions × 7 distances = 56 planes
#   Directions: N, NE, E, SE, S, SW, W, NW (from current player's perspective)
#   Distance: 1-7 squares
#
# Planes 56-63: Knight moves (8 possible L-shapes)
#
# Planes 64-72: Underpromotions (promote to N, B, or R — not Q)
#   3 directions (capture-left, forward, capture-right) × 3 pieces = 9 planes
#   Queen promotions reuse the normal queen-move planes (distance=1, forward/diagonal)

DIRECTIONS = [
    (1, 0),   # N  — rank+1
    (1, 1),   # NE — rank+1, file+1
    (0, 1),   # E  — file+1
    (-1, 1),  # SE — rank-1, file+1
    (-1, 0),  # S  — rank-1
    (-1, -1), # SW — rank-1, file-1
    (0, -1),  # W  — file-1
    (1, -1),  # NW — rank+1, file-1
]

KNIGHT_DELTAS = [
    (2, 1), (2, -1), (-2, 1), (-2, -1),
    (1, 2), (1, -2), (-1, 2), (-1, -2),
]

UNDERPROMO_PIECES = [chess.KNIGHT, chess.BISHOP, chess.ROOK]
# Underpromo directions in (dr, dc) from current player's perspective (board always flipped to player's POV)
# Pawn always moves "up" (dr=+1). Three options: capture-left, straight, capture-right
UNDERPROMO_DIRS = [(1, -1), (1, 0), (1, 1)]  # capture-left, straight, capture-right


def _queen_plane(dr: int, dc: int) -> int:
    """Get plane index for a queen-style move with delta (dr, dc) per step."""
    dir_idx = DIRECTIONS.index((dr, dc))
    return dir_idx  # Plane 0-7 for distance 1; we multiply by distance offset below


def move_to_index(move: chess.Move, board: chess.Board) -> int:
    """
    Convert a chess move to a policy index (0-4671).

    Args:
        move: The move to encode
        board: Current board position (needed for perspective flip and context)

    Returns:
        Integer index in [0, 4671]
    """
    flip = board.turn == chess.BLACK

    from_sq = move.from_square
    to_sq = move.to_square

    from_rank = chess.square_rank(from_sq)
    from_file = chess.square_file(from_sq)
    to_rank = chess.square_rank(to_sq)
    to_file = chess.square_file(to_sq)

    # Flip coordinates for Black's perspective
    if flip:
        from_rank = 7 - from_rank
        to_rank = 7 - to_rank

    dr = to_rank - from_rank
    dc = to_file - from_file

    source_square_idx = from_rank * 8 + from_file  # 0-63

    # --- Underpromotion? ---
    if move.promotion is not None and move.promotion != chess.QUEEN:
        piece_idx = UNDERPROMO_PIECES.index(move.promotion)
        # Find which underpromo direction
        dir_idx = UNDERPROMO_DIRS.index((dr, dc))
        plane = 64 + piece_idx * 3 + dir_idx
        return source_square_idx * 73 + plane

    # --- Knight move? ---
    if board.piece_at(from_sq) is not None and board.piece_at(from_sq).piece_type == chess.KNIGHT:
        knight_idx = KNIGHT_DELTAS.index((dr, dc))
        plane = 56 + knight_idx
        return source_square_idx * 73 + plane

    # --- Queen-style move (includes pawns) ---
    distance = max(abs(dr), abs(dc))
    unit_dr = dr // distance if dr != 0 else 0
    unit_dc = dc // distance if dc != 0 else 0
    dir_idx = DIRECTIONS.index((unit_dr, unit_dc))
    plane = dir_idx * 7 + (distance - 1)  # 0-55
    return source_square_idx * 73 + plane


def index_to_move(index: int, board: chess.Board) -> chess.Move:
    """
    Convert a policy index back to a chess.Move.

    This is the reverse of move_to_index. Used when the network outputs
    a policy distribution and we need to pick the actual move to play.

    Returns chess.Move.null() if the index doesn't map to a legal move.
    """
    flip = board.turn == chess.BLACK

    source_square_idx = index // 73
    plane = index % 73

    from_rank = source_square_idx // 8
    from_file = source_square_idx % 8

    # Undo the perspective flip
    actual_from_rank = (7 - from_rank) if flip else from_rank
    from_sq = chess.square(from_file, actual_from_rank)

    promotion = None

    if plane >= 64:
        # Underpromotion
        plane_offset = plane - 64
        piece_idx = plane_offset // 3
        dir_idx = plane_offset % 3
        promotion = UNDERPROMO_PIECES[piece_idx]
        dr, dc = UNDERPROMO_DIRS[dir_idx]
        to_rank = from_rank + dr
        to_file = from_file + dc
    elif plane >= 56:
        # Knight move
        knight_idx = plane - 56
        dr, dc = KNIGHT_DELTAS[knight_idx]
        to_rank = from_rank + dr
        to_file = from_file + dc
    else:
        # Queen-style move
        dir_idx = plane // 7
        distance = (plane % 7) + 1
        unit_dr, unit_dc = DIRECTIONS[dir_idx]
        to_rank = from_rank + unit_dr * distance
        to_file = from_file + unit_dc * distance

    if not (0 <= to_rank <= 7 and 0 <= to_file <= 7):
        return chess.Move.null()

    actual_to_rank = (7 - to_rank) if flip else to_rank
    to_sq = chess.square(to_file, actual_to_rank)

    # Queen promotion: if pawn reaches back rank and no underpromotion specified
    piece = board.piece_at(from_sq)
    if piece is not None and piece.piece_type == chess.PAWN and promotion is None:
        back_rank = 7 if board.turn == chess.WHITE else 0
        if chess.square_rank(to_sq) == back_rank:
            promotion = chess.QUEEN

    return chess.Move(from_sq, to_sq, promotion=promotion)


def get_legal_move_indices(board: chess.Board) -> list[int]:
    """Return the policy indices of all legal moves in the current position."""
    return [move_to_index(move, board) for move in board.legal_moves]


def policy_to_moves(policy: np.ndarray, board: chess.Board, top_k: int = 10) -> list[tuple[chess.Move, float]]:
    """
    Given a raw policy array (4672 values), return the top-k legal moves
    with their probabilities (after masking illegal moves and softmax).

    Used by the visualization app to draw move arrows.
    """
    import torch
    legal_indices = get_legal_move_indices(board)
    if not legal_indices:
        return []

    # Mask: set illegal moves to -inf so they get ~0 probability
    masked = np.full(4672, -1e9, dtype=np.float32)
    for idx in legal_indices:
        masked[idx] = policy[idx]

    # Softmax
    exp = np.exp(masked - masked[legal_indices].max())
    exp_sum = exp[legal_indices].sum()
    probs = np.zeros(4672, dtype=np.float32)
    for idx in legal_indices:
        probs[idx] = exp[idx] / exp_sum

    # Sort by probability
    sorted_indices = sorted(legal_indices, key=lambda i: probs[i], reverse=True)
    results = []
    for idx in sorted_indices[:top_k]:
        move = index_to_move(idx, board)
        results.append((move, float(probs[idx])))
    return results


Overwriting /content/chess-nn/chess_nn/move_encoding.py


: 

In [31]:
%%writefile /content/chess-nn/chess_nn/model.py
"""
Neural network architecture: a CNN with residual blocks and two output heads.

Architecture summary:
  Input (18, 8, 8)
    → Initial conv layer
    → 5 residual blocks  (the "body" — extracts chess patterns)
    → Policy head        (outputs probabilities over 4672 moves)
    → Value head         (outputs a single number: who's winning)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from config import NUM_RESIDUAL_BLOCKS, NUM_FILTERS, INPUT_PLANES, POLICY_OUTPUT_SIZE


class ResidualBlock(nn.Module):
    """
    A residual block: two conv layers with a skip connection.

    The skip connection adds the input directly to the output, which lets
    the network learn "corrections" rather than full transformations.
    This is what makes deep networks trainable — without it, gradients
    vanish before reaching early layers.
    """
    def __init__(self, num_filters: int):
        super().__init__()
        self.conv1 = nn.Conv2d(num_filters, num_filters, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(num_filters)
        self.conv2 = nn.Conv2d(num_filters, num_filters, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(num_filters)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x = F.relu(x + residual)  # Skip connection: add input back in
        return x


class ChessNet(nn.Module):
    """
    The full chess neural network.

    Input:  (batch, 18, 8, 8) board tensor
    Output: policy logits (batch, 4672) and value (batch, 1)
    """
    def __init__(self):
        super().__init__()

        # Initial conv: expand from 18 input planes to NUM_FILTERS feature maps
        self.input_conv = nn.Conv2d(INPUT_PLANES, NUM_FILTERS, kernel_size=3, padding=1, bias=False)
        self.input_bn = nn.BatchNorm2d(NUM_FILTERS)

        # Body: stack of residual blocks
        self.residual_blocks = nn.ModuleList([
            ResidualBlock(NUM_FILTERS) for _ in range(NUM_RESIDUAL_BLOCKS)
        ])

        # Policy head: shrink filters → flatten → predict move probabilities
        self.policy_conv = nn.Conv2d(NUM_FILTERS, 2, kernel_size=1, bias=False)
        self.policy_bn = nn.BatchNorm2d(2)
        self.policy_fc = nn.Linear(2 * 8 * 8, POLICY_OUTPUT_SIZE)

        # Value head: shrink filters → flatten → predict game outcome
        self.value_conv = nn.Conv2d(NUM_FILTERS, 1, kernel_size=1, bias=False)
        self.value_bn = nn.BatchNorm2d(1)
        self.value_fc1 = nn.Linear(1 * 8 * 8, 256)
        self.value_fc2 = nn.Linear(256, 1)

        # Dropout applied in heads only — residual blocks use BN which already regularizes
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x):
        # Body
        x = F.relu(self.input_bn(self.input_conv(x)))
        for block in self.residual_blocks:
            x = block(x)

        # Policy head
        p = F.relu(self.policy_bn(self.policy_conv(x)))
        p = p.view(p.size(0), -1)
        p = self.dropout(p)
        p = self.policy_fc(p)

        # Value head
        v = F.relu(self.value_bn(self.value_conv(x)))
        v = v.view(v.size(0), -1)
        v = F.relu(self.dropout(self.value_fc1(v)))
        v = torch.tanh(self.value_fc2(v))

        return p, v

    def get_all_activations(self, x: torch.Tensor) -> dict:
        """
        Run a forward pass and capture the output of every major layer.
        Returns a dict of name → numpy array for the network visualizer.
        """
        captured = {}
        hooks = []

        def make_hook(name):
            def fn(module, inp, out):
                captured[name] = out.detach().cpu().numpy()
            return fn

        hooks.append(self.input_conv.register_forward_hook(make_hook("input_conv")))
        for i, block in enumerate(self.residual_blocks):
            hooks.append(block.register_forward_hook(make_hook(f"res_block_{i}")))
        hooks.append(self.policy_conv.register_forward_hook(make_hook("policy_conv")))
        hooks.append(self.value_conv.register_forward_hook(make_hook("value_conv")))

        with torch.no_grad():
            policy_logits, value = self.forward(x)

        for h in hooks:
            h.remove()

        captured["policy_logits"] = policy_logits.detach().cpu().numpy()
        captured["value"] = value.detach().cpu().numpy()
        return captured

    def get_activations(self, x: torch.Tensor, layer_index: int = 0) -> torch.Tensor:
        """
        Run a forward pass and capture the output of a specific residual block.
        Used by the visualization app to show what the network "pays attention to."

        Returns: (8, 8) heatmap averaged across all filters
        """
        activations = {}

        def hook_fn(module, input, output):
            activations['out'] = output.detach()

        handle = self.residual_blocks[layer_index].register_forward_hook(hook_fn)
        with torch.no_grad():
            self.forward(x)
        handle.remove()

        # Average across the filter dimension: (batch, 128, 8, 8) → (8, 8)
        return activations['out'].squeeze(0).mean(dim=0)


Overwriting /content/chess-nn/chess_nn/model.py


: 

In [32]:
%%writefile /content/chess-nn/chess_nn/dataset.py
"""
Dataset: convert PGN games into tensors the model can train on.

For each game, we walk through every position and create a training example:
  - Input:  board tensor (18, 8, 8)
  - Policy target: the move that was actually played (as an index 0-4671)
  - Value target:  the final game result from the current player's perspective
                   +1 = current player won, -1 = lost, 0 = draw

We save in chunks of 100k positions as .npz files to keep memory usage low.
"""

import os
import sys
import bisect
import numpy as np
import chess
import chess.pgn
from tqdm import tqdm
import torch
from torch.utils.data import Dataset, DataLoader, Sampler, random_split

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import PROCESSED_DATA_DIR, TRAIN_SPLIT, VAL_SPLIT, BATCH_SIZE, POLICY_OUTPUT_SIZE
from chess_nn.board_encoding import board_to_tensor
from chess_nn.move_encoding import move_to_index, get_legal_move_indices

CHUNK_SIZE = 100_000


def result_to_value(result: str, turn: bool) -> float:
    """Convert PGN result string to a value from the current player's perspective."""
    if result == "1-0":
        return 1.0 if turn == chess.WHITE else -1.0
    elif result == "0-1":
        return -1.0 if turn == chess.WHITE else 1.0
    else:
        return 0.0  # Draw


def process_games(pgn_path: str, output_dir: str = None) -> list[str]:
    """
    Read a PGN file, convert every position to training examples, save as .npz chunks.
    Returns list of paths to the saved chunk files.
    """
    if output_dir is None:
        output_dir = PROCESSED_DATA_DIR
    os.makedirs(output_dir, exist_ok=True)

    boards, policies, values, masks = [], [], [], []
    chunk_idx = 0
    chunk_paths = []
    total_positions = 0

    print(f"Processing games from: {pgn_path}")

    games_seen = 0
    games_kept = 0

    with open(pgn_path) as pgn_file:
        pbar = tqdm(desc="Processing", unit="pos", dynamic_ncols=True)

        while True:
            game = chess.pgn.read_game(pgn_file)
            if game is None:
                break

            games_seen += 1
            result = game.headers.get("Result", "*")
            if result not in ("1-0", "0-1", "1/2-1/2"):
                continue

            board = game.board()
            for move in game.mainline_moves():
                board_tensor = board_to_tensor(board)
                move_idx = move_to_index(move, board)
                value = result_to_value(result, board.turn)

                legal_mask = np.zeros(POLICY_OUTPUT_SIZE, dtype=bool)
                for idx in get_legal_move_indices(board):
                    legal_mask[idx] = True

                boards.append(board_tensor)
                policies.append(move_idx)
                values.append(value)
                masks.append(legal_mask)

                board.push(move)
                total_positions += 1
                pbar.update(1)
                pbar.set_postfix(games=games_kept, chunk=chunk_idx)

                if len(boards) >= CHUNK_SIZE:
                    path = _save_chunk(boards, policies, values, masks, output_dir, chunk_idx)
                    chunk_paths.append(path)
                    chunk_idx += 1
                    boards, policies, values, masks = [], [], [], []

            games_kept += 1

        pbar.close()

    if boards:
        path = _save_chunk(boards, policies, values, masks, output_dir, chunk_idx)
        chunk_paths.append(path)

    print(f"Total positions: {total_positions:,} across {len(chunk_paths)} chunks")
    return chunk_paths


def _save_chunk(boards, policies, values, masks, output_dir, chunk_idx):
    path = os.path.join(output_dir, f"chunk_{chunk_idx:04d}.npz")
    np.savez_compressed(
        path,
        boards=np.array(boards, dtype=np.float32),
        policies=np.array(policies, dtype=np.int64),
        values=np.array(values, dtype=np.float32),
        legal_masks=np.packbits(np.array(masks, dtype=bool), axis=1),
    )
    print(f"\nSaved chunk {chunk_idx} ({len(boards):,} positions) → {path}")
    return path


class ChessDataset(Dataset):
    """
    PyTorch Dataset that loads from .npz chunk files — one chunk at a time.

    Instead of loading all positions into RAM at once (which can use 5+ GB),
    this scans chunk sizes upfront then loads one chunk (~450 MB) on demand.
    Each DataLoader worker maintains its own single-chunk cache, so peak RAM
    is roughly: (num_workers + 1) × chunk_size × ~4.5 KB per position.
    """

    def __init__(self, chunk_paths: list[str]):
        self.chunk_paths = chunk_paths

        # Scan sizes only — no arrays loaded yet
        self.chunk_sizes = []
        for path in chunk_paths:
            with np.load(path) as f:
                self.chunk_sizes.append(len(f["boards"]))

        # Prefix sums so we can bisect-search chunk_idx for any global idx
        self.offsets = [0]
        for sz in self.chunk_sizes:
            self.offsets.append(self.offsets[-1] + sz)

        print(f"Dataset: {self.offsets[-1]:,} positions across {len(chunk_paths)} chunks (lazy)")

        # Single-chunk cache per worker process
        self._cached_chunk_idx = None
        self._cached_data = None

    def __len__(self):
        return self.offsets[-1]

    def __getitem__(self, idx):
        # Find which chunk owns this index
        chunk_idx = bisect.bisect_right(self.offsets, idx) - 1
        chunk_idx = max(0, min(chunk_idx, len(self.chunk_paths) - 1))
        local_idx = idx - self.offsets[chunk_idx]

        # Load chunk into RAM if it's not already cached
        if chunk_idx != self._cached_chunk_idx:
            self._cached_data = dict(np.load(self.chunk_paths[chunk_idx]))
            self._cached_chunk_idx = chunk_idx

        data = self._cached_data
        mask = np.unpackbits(data["legal_masks"][local_idx])[:POLICY_OUTPUT_SIZE].astype(bool)
        return (
            torch.from_numpy(data["boards"][local_idx].copy()),
            torch.tensor(int(data["policies"][local_idx]), dtype=torch.long),
            torch.tensor(float(data["values"][local_idx]), dtype=torch.float32),
            torch.from_numpy(mask),
        )


class ChunkBatchSampler(Sampler):
    """
    Yield batches where every index belongs to the same .npz chunk.

    With shuffle=False (validation), items are ordered to be sequential within chunks.
    With shuffle=True (training), both chunk order and within-chunk order are randomised
    each epoch — giving good diversity without cache thrashing.
    """

    def __init__(self, subset, batch_size: int, shuffle: bool = True):
        from collections import defaultdict
        dataset = subset.dataset
        chunk_groups: dict[int, list[int]] = defaultdict(list)
        for local_idx, gi in enumerate(subset.indices):
            ci = bisect.bisect_right(dataset.offsets, gi) - 1
            chunk_groups[ci].append(local_idx)
        self._groups = list(chunk_groups.values())
        self._batch_size = batch_size
        self._shuffle = shuffle

    def __iter__(self):
        import random
        groups = [g[:] for g in self._groups]
        if self._shuffle:
            random.shuffle(groups)
            groups = [random.sample(g, len(g)) for g in groups]
        for group in groups:
            for i in range(0, len(group), self._batch_size):
                batch = group[i:i + self._batch_size]
                if batch:
                    yield batch

    def __len__(self) -> int:
        return sum(
            (len(g) + self._batch_size - 1) // self._batch_size
            for g in self._groups
        )


def make_dataloaders(chunk_paths: list[str]):
    """Split data into train/val/test DataLoaders."""
    dataset = ChessDataset(chunk_paths)
    n = len(dataset)
    n_train = int(n * TRAIN_SPLIT)
    n_val = int(n * VAL_SPLIT)
    n_test = n - n_train - n_val

    train_set, val_set, test_set = random_split(
        dataset, [n_train, n_val, n_test],
        generator=torch.Generator().manual_seed(42)
    )

    train_sampler = ChunkBatchSampler(train_set, BATCH_SIZE, shuffle=True)
    val_sampler   = ChunkBatchSampler(val_set,   BATCH_SIZE, shuffle=False)
    test_sampler  = ChunkBatchSampler(test_set,  BATCH_SIZE, shuffle=False)
    train_loader = DataLoader(train_set, batch_sampler=train_sampler, num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_set,   batch_sampler=val_sampler,   num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_sampler=test_sampler,  num_workers=4, pin_memory=True)

    print(f"Train: {len(train_set):,} | Val: {len(val_set):,} | Test: {len(test_set):,}")
    return train_loader, val_loader, test_loader


if __name__ == "__main__":
    import glob
    from config import RAW_DATA_DIR

    pgn_files = sorted(glob.glob(os.path.join(RAW_DATA_DIR, "*.pgn")))
    if not pgn_files:
        print(f"No PGN files found in {RAW_DATA_DIR}. Run data/download_data.py first.")
        sys.exit(1)

    for pgn_path in pgn_files:
        process_games(pgn_path)


Overwriting /content/chess-nn/chess_nn/dataset.py


: 

In [33]:
%%writefile /content/chess-nn/chess_nn/train.py
"""
Training loop: teach the network to predict moves and evaluate positions.

Two losses are combined:
  - Policy loss: cross-entropy between predicted move probabilities and the actual move played
                 (like teaching a student: "in this position, this move was correct")
  - Value loss:  MSE between predicted win probability and actual game outcome
                 (like teaching: "this position eventually led to a win/loss/draw")

Total loss = policy_loss + 0.5 * value_loss
"""

import glob
import os
import sys
import time
import torch
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import (
    DEVICE, LEARNING_RATE, WEIGHT_DECAY, NUM_EPOCHS,
    GRADIENT_CLIP, VALUE_LOSS_WEIGHT, CHECKPOINT_DIR
)
from chess_nn.model import ChessNet
from chess_nn.utils import save_checkpoint, load_checkpoint, log_training_step
from chess_nn.dataset import make_dataloaders

WARMUP_STEPS = 1000     # Ramp LR from 0 → LEARNING_RATE over first 1000 steps
LABEL_SMOOTHING = 0.1   # Fraction of probability redistributed uniformly across legal moves
DIRICHLET_ALPHA = 0.3   # Concentration — low = sparse/focused, AlphaZero uses 0.3 for chess
DIRICHLET_EPS   = 0.25  # Fraction of target replaced with Dirichlet noise during training


def get_lr_scale(step: int) -> float:
    if step < WARMUP_STEPS:
        return step / WARMUP_STEPS
    return 1.0


def make_policy_targets(policy_targets, legal_masks, add_noise: bool):
    """
    Build soft policy targets over legal moves only.

    Without noise: (1-smooth)*one_hot + smooth*uniform_over_legal
    With noise:    mix in Dirichlet noise so the model sees that less common
                   legal moves sometimes get played (human error / rare lines).

    Why Dirichlet?  α=0.3 produces sparse samples — most mass on 2-3 moves but
    nonzero everywhere legal.  This mimics how humans sometimes choose second-best
    moves without being purely random.
    """
    B, N = legal_masks.shape
    n_legal = legal_masks.float().sum(dim=1, keepdim=True).clamp(min=1)
    uniform_legal = legal_masks.float() / n_legal
    one_hot = F.one_hot(policy_targets, num_classes=N).float()
    soft = (1.0 - LABEL_SMOOTHING) * one_hot + LABEL_SMOOTHING * uniform_legal

    if add_noise:
        concentration = torch.full((N,), DIRICHLET_ALPHA)  # CPU — MPS lacks _sample_dirichlet
        noise = torch.distributions.Dirichlet(concentration).sample((B,)).to(legal_masks.device)
        noise = noise * legal_masks.float()
        noise = noise / noise.sum(dim=1, keepdim=True).clamp(min=1e-8)
        soft = (1.0 - DIRICHLET_EPS) * soft + DIRICHLET_EPS * noise

    return soft


def train(chunk_paths: list[str], resume: bool = False):
    print(f"Training on device: {DEVICE}")

    model = ChessNet().to(DEVICE)
    optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    start_epoch = 0
    if resume:
        epoch_ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "epoch_*.pt")))
        if epoch_ckpts:
            start_epoch, _ = load_checkpoint(epoch_ckpts[-1], model, optimizer, scheduler)
            print(f"Resuming from epoch {start_epoch} → continuing to epoch {NUM_EPOCHS}")
        else:
            print("No epoch checkpoints found — starting from scratch")

    if start_epoch >= NUM_EPOCHS:
        print(f"Already completed {NUM_EPOCHS} epochs. Done.")
        return model

    train_loader, val_loader, _ = make_dataloaders(chunk_paths)

    best_val_loss = float("inf")
    global_step = start_epoch * len(train_loader)  # Approximate step count for LR warmup

    # How often to print a status line (every 10% of an epoch, or every 100 batches)
    total_train_batches = len(train_loader)
    print_every = max(100, total_train_batches // 10)

    for epoch in range(start_epoch + 1, NUM_EPOCHS + 1):
        epoch_start = time.time()

        print(f"\n{'='*54}")
        print(f"  Epoch {epoch} / {NUM_EPOCHS}")
        if epoch == 1:
            print("  Note: first ~20 batches are slow while MPS warms up.")
        print(f"{'='*54}")

        # --- Training ---
        model.train()
        train_policy_loss = 0.0
        train_value_loss  = 0.0
        n_batches = 0
        MPS_WARMUP = 20  # batches to skip before trusting the ETA

        for boards, policy_targets, value_targets, legal_masks in train_loader:
            boards         = boards.to(DEVICE)
            policy_targets = policy_targets.to(DEVICE)
            value_targets  = value_targets.to(DEVICE)
            legal_masks    = legal_masks.to(DEVICE)

            if global_step < WARMUP_STEPS:
                scale = get_lr_scale(global_step)
                for pg in optimizer.param_groups:
                    pg["lr"] = LEARNING_RATE * scale

            policy_logits, value_pred = model(boards)
            masked_logits = policy_logits.masked_fill(~legal_masks, float("-inf"))

            soft_targets = make_policy_targets(policy_targets, legal_masks, add_noise=True)
            log_probs = F.log_softmax(masked_logits, dim=1).masked_fill(~legal_masks, 0.0)
            policy_loss = -(soft_targets * log_probs).sum(dim=1).mean()
            value_loss = F.mse_loss(value_pred.squeeze(1), value_targets)
            total_loss = policy_loss + VALUE_LOSS_WEIGHT * value_loss

            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()

            train_policy_loss += policy_loss.item()
            train_value_loss  += value_loss.item()
            n_batches  += 1
            global_step += 1

            if global_step % 100 == 0:
                log_training_step(
                    epoch, global_step,
                    policy_loss.item(), value_loss.item(), total_loss.item()
                )

            if n_batches % print_every == 0 or n_batches == total_train_batches:
                pct     = n_batches / total_train_batches * 100
                elapsed = time.time() - epoch_start
                avg_policy = train_policy_loss / n_batches
                avg_value  = train_value_loss  / n_batches
                lr_now     = optimizer.param_groups[0]["lr"]

                # Only show ETA after MPS has warmed up
                if n_batches > MPS_WARMUP:
                    secs_per_batch = elapsed / n_batches
                    remaining_batches = total_train_batches - n_batches
                    eta_secs = secs_per_batch * remaining_batches
                    eta_m, eta_s = divmod(int(eta_secs), 60)
                    eta_str = f"ETA {eta_m}m {eta_s:02d}s"
                else:
                    eta_str = "ETA warming up..."

                e_m, e_s = divmod(int(elapsed), 60)
                print(
                    f"  {n_batches:>5}/{total_train_batches}  [{pct:5.1f}%]"
                    f"  policy {avg_policy:.3f}  value {avg_value:.3f}"
                    f"  lr {lr_now:.5f}  |  {e_m}m {e_s:02d}s elapsed  {eta_str}"
                )

        scheduler.step()

        # --- Validation ---
        print("\n  Running validation...")
        model.eval()
        val_policy_loss = 0.0
        val_value_loss  = 0.0
        correct_top1 = 0
        correct_top5 = 0
        total = 0

        with torch.no_grad():
            for boards, policy_targets, value_targets, legal_masks in val_loader:
                boards         = boards.to(DEVICE)
                policy_targets = policy_targets.to(DEVICE)
                value_targets  = value_targets.to(DEVICE)
                legal_masks    = legal_masks.to(DEVICE)

                policy_logits, value_pred = model(boards)
                masked_logits = policy_logits.masked_fill(~legal_masks, float("-inf"))
                soft_targets  = make_policy_targets(policy_targets, legal_masks, add_noise=False)
                log_probs = F.log_softmax(masked_logits, dim=1).masked_fill(~legal_masks, 0.0)
                val_policy_loss += (-(soft_targets * log_probs).sum(dim=1).mean()).item()
                val_value_loss += F.mse_loss(value_pred.squeeze(1), value_targets).item()

                top5_pred = masked_logits.topk(5, dim=1).indices
                correct_top1 += (top5_pred[:, 0] == policy_targets).sum().item()
                correct_top5 += (top5_pred == policy_targets.unsqueeze(1)).any(dim=1).sum().item()
                total += len(policy_targets)

        n_val       = len(val_loader)
        avg_val_policy = val_policy_loss / n_val
        avg_val_value  = val_value_loss  / n_val
        top1_acc = correct_top1 / total * 100
        top5_acc = correct_top5 / total * 100

        elapsed = time.time() - epoch_start
        e_m, e_s = divmod(int(elapsed), 60)

        print(f"\n  --- Epoch {epoch} results ---")
        print(f"  Time this epoch    : {e_m}m {e_s:02d}s")
        print(f"  Policy loss        : {avg_val_policy:.4f}  (lower = better move picks)")
        print(f"  Value loss         : {avg_val_value:.4f}  (lower = better win-rate estimate)")
        print(f"  Move accuracy top1 : {top1_acc:.1f}%   top5: {top5_acc:.1f}%")

        remaining_epochs = NUM_EPOCHS - epoch
        if remaining_epochs > 0:
            r_m, r_s = divmod(int(elapsed * remaining_epochs), 60)
            print(f"  ETA for remaining  : ~{r_m}m {r_s:02d}s  ({remaining_epochs} epochs left)")

        # Save best model
        val_total = avg_val_policy + VALUE_LOSS_WEIGHT * avg_val_value
        if val_total < best_val_loss:
            best_val_loss = val_total
            save_checkpoint(model, optimizer, epoch, val_total, "best_model.pt", scheduler)
            print(f"  >> New best model saved  (combined loss {val_total:.4f})")

        save_checkpoint(model, optimizer, epoch, val_total, f"epoch_{epoch:02d}.pt", scheduler)

    print("\nTraining complete.")
    return model


if __name__ == "__main__":
    import glob
    chunk_paths = sorted(glob.glob(os.path.join(
        os.path.dirname(os.path.dirname(os.path.abspath(__file__))),
        "data", "processed", "chunk_*.npz"
    )))
    if not chunk_paths:
        print("No processed data found. Run data/download_data.py first, then chess_nn/dataset.py.")
        sys.exit(1)
    train(chunk_paths)


Overwriting /content/chess-nn/chess_nn/train.py


: 

In [34]:
%%writefile /content/chess-nn/chess_nn/utils.py
import os
import json
import torch
from config import CHECKPOINT_DIR, LOG_DIR


def get_device():
    import torch
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def count_parameters(model: torch.nn.Module) -> int:
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {total:,}")
    return total


def save_checkpoint(model, optimizer, epoch: int, loss: float,
                    filename: str = "checkpoint.pt", scheduler=None):
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    path = os.path.join(CHECKPOINT_DIR, filename)
    data = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "loss": loss,
    }
    if scheduler is not None:
        data["scheduler_state"] = scheduler.state_dict()
    torch.save(data, path)
    print(f"Checkpoint saved: {path}")
    return path


def load_checkpoint(path: str, model, optimizer=None, scheduler=None):
    checkpoint = torch.load(path, map_location=get_device(), weights_only=False)
    model.load_state_dict(checkpoint["model_state"])
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state"])
    if scheduler is not None and "scheduler_state" in checkpoint:
        scheduler.load_state_dict(checkpoint["scheduler_state"])
    print(f"Loaded checkpoint from epoch {checkpoint['epoch']} (loss={checkpoint['loss']:.4f})")
    return checkpoint["epoch"], checkpoint["loss"]


def log_training_step(epoch: int, step: int, policy_loss: float, value_loss: float, total_loss: float):
    """Append a training step to the JSON log so the viz app can read it live."""
    os.makedirs(LOG_DIR, exist_ok=True)
    log_path = os.path.join(LOG_DIR, "training_log.json")

    entry = {
        "epoch": epoch,
        "step": step,
        "policy_loss": round(policy_loss, 6),
        "value_loss": round(value_loss, 6),
        "total_loss": round(total_loss, 6),
    }

    # Read existing entries, append, write back
    entries = []
    if os.path.exists(log_path):
        with open(log_path) as f:
            try:
                entries = json.load(f)
            except json.JSONDecodeError:
                entries = []
    entries.append(entry)
    with open(log_path, "w") as f:
        json.dump(entries, f)


Overwriting /content/chess-nn/chess_nn/utils.py


: 

In [35]:
%%writefile /content/chess-nn/chess_nn/evaluate.py
"""
Evaluate how well the trained model plays chess.

Metrics:
  - Top-1 accuracy: did the model pick the exact same move a human played?
  - Top-5 accuracy: was the human's move in the model's top 5 choices?
  - Value MSE: how close was the win probability estimate to the real outcome?
  - Play test: win rate vs. a random-move player
"""

import sys
import os
import random
import torch
import chess
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import DEVICE
from chess_nn.model import ChessNet
from chess_nn.board_encoding import board_to_tensor
from chess_nn.move_encoding import move_to_index, get_legal_move_indices, policy_to_moves
from chess_nn.utils import load_checkpoint


def evaluate_dataset(model, data_loader):
    """Compute top-1, top-5 accuracy and value MSE on a dataset split."""
    model.eval()
    correct_top1 = correct_top5 = total = 0
    value_mse_sum = 0.0

    with torch.no_grad():
        for boards, policy_targets, value_targets in data_loader:
            boards = boards.to(DEVICE)
            policy_targets = policy_targets.to(DEVICE)
            value_targets = value_targets.to(DEVICE)

            policy_logits, value_pred = model(boards)

            top5 = policy_logits.topk(5, dim=1).indices
            correct_top1 += (top5[:, 0] == policy_targets).sum().item()
            correct_top5 += (top5 == policy_targets.unsqueeze(1)).any(dim=1).sum().item()
            total += len(policy_targets)

            value_mse_sum += ((value_pred.squeeze(1) - value_targets) ** 2).sum().item()

    return {
        "top1_acc": correct_top1 / total * 100,
        "top5_acc": correct_top5 / total * 100,
        "value_mse": value_mse_sum / total,
    }


def select_move(model, board: chess.Board, temperature: float = 1.0) -> chess.Move:
    """
    Pick a move using the model's policy output.

    temperature=0: always pick the highest-probability move (greedy)
    temperature=1: sample proportionally to probabilities (more variety)
    """
    device = next(model.parameters()).device
    tensor = torch.from_numpy(board_to_tensor(board)).unsqueeze(0).to(device)
    with torch.no_grad():
        policy_logits, value = model(tensor)

    policy = policy_logits.squeeze(0).cpu().numpy()
    legal_indices = get_legal_move_indices(board)

    # Mask illegal moves
    masked = np.full(4672, -1e9, dtype=np.float32)
    for idx in legal_indices:
        masked[idx] = policy[idx]

    if temperature == 0:
        chosen_idx = legal_indices[np.argmax([masked[i] for i in legal_indices])]
    else:
        # Apply temperature, then softmax
        scaled = np.array([masked[i] / temperature for i in legal_indices])
        scaled -= scaled.max()
        probs = np.exp(scaled)
        probs /= probs.sum()
        chosen_idx = np.random.choice(legal_indices, p=probs)

    from chess_nn.move_encoding import index_to_move
    return index_to_move(chosen_idx, board)


def play_vs_random(model, n_games: int = 50, temperature: float = 0.5) -> dict:
    """Play n games against a random-move opponent. Model plays both colors."""
    wins = draws = losses = 0

    for game_idx in range(n_games):
        board = chess.Board()
        model_is_white = (game_idx % 2 == 0)

        while not board.is_game_over():
            if board.turn == chess.WHITE and model_is_white:
                move = select_move(model, board, temperature)
            elif board.turn == chess.BLACK and not model_is_white:
                move = select_move(model, board, temperature)
            else:
                move = random.choice(list(board.legal_moves))
            board.push(move)

        result = board.result()
        if (result == "1-0" and model_is_white) or (result == "0-1" and not model_is_white):
            wins += 1
        elif result == "1/2-1/2":
            draws += 1
        else:
            losses += 1

    return {"wins": wins, "draws": draws, "losses": losses, "win_rate": wins / n_games * 100}


if __name__ == "__main__":
    import glob
    from chess_nn.dataset import make_dataloaders

    chunk_paths = sorted(glob.glob(
        os.path.join(os.path.dirname(os.path.dirname(__file__)), "data", "processed", "chunk_*.npz")
    ))

    model = ChessNet().to(DEVICE)
    checkpoint_path = os.path.join(
        os.path.dirname(os.path.dirname(__file__)), "checkpoints", "best_model.pt"
    )
    load_checkpoint(checkpoint_path, model)

    _, val_loader, test_loader = make_dataloaders(chunk_paths)

    print("Evaluating on test set...")
    metrics = evaluate_dataset(model, test_loader)
    print(f"  Top-1 accuracy: {metrics['top1_acc']:.1f}%")
    print(f"  Top-5 accuracy: {metrics['top5_acc']:.1f}%")
    print(f"  Value MSE:      {metrics['value_mse']:.4f}")

    print("\nPlaying 20 games vs. random mover...")
    results = play_vs_random(model, n_games=20)
    print(f"  Wins: {results['wins']}  Draws: {results['draws']}  Losses: {results['losses']}")
    print(f"  Win rate: {results['win_rate']:.0f}%")


Overwriting /content/chess-nn/chess_nn/evaluate.py


: 

In [36]:
%%writefile /content/chess-nn/chess_nn/mcts.py
"""
Monte Carlo Tree Search (MCTS) for chess.

The tree is made of Nodes. Each Node represents one board position.
We grow the tree by repeatedly running 4 steps:
  1. Select   — walk down using UCB formula until we hit an unvisited node
  2. Expand   — ask the neural network what it thinks of this new position
  3. Backup   — send the value back up to all ancestors
  4. (repeat N times, then pick the most-visited move)
"""

import math
import chess
import torch
import numpy as np

from chess_nn.board_encoding import board_to_tensor
from chess_nn.move_encoding import move_to_index, index_to_move, get_legal_move_indices

# Exploration constant: higher = more exploration, lower = more exploitation.
# AlphaZero uses ~1.4. Think of it as "how curious is the searcher?"
C_PUCT = 1.4


class Node:
    """
    One node in the search tree = one chess position.

    Each node tracks:
      - How many times we've visited it (N)
      - The total value accumulated from all visits (W)
      - The average value Q = W / N
      - The prior probability P from the neural network (how likely this move looked)
      - Its children (one per legal move)
    """

    def __init__(self, prior: float = 0.0):
        self.N = 0        # Visit count
        self.W = 0.0      # Total value (sum of backpropagated values)
        self.P = prior    # Prior probability from network policy head

        # Dict of {chess.Move: Node} — populated lazily when this node is expanded
        self.children: dict[chess.Move, "Node"] = {}
        self.is_expanded = False

    @property
    def Q(self) -> float:
        """Average value. 0 if never visited (optimistic for unexplored nodes)."""
        return self.W / self.N if self.N > 0 else 0.0

    def ucb_score(self, parent_visits: int) -> float:
        """
        Upper Confidence Bound formula:
          Q  = exploitation: average reward seen from this node
          U  = exploration bonus: high when P is large or visits are low

        The sqrt term ensures every node gets visited eventually,
        but nodes with high prior P or high Q get visited more.
        """
        U = C_PUCT * self.P * math.sqrt(parent_visits) / (1 + self.N)
        return self.Q + U

    def best_child(self) -> tuple[chess.Move, "Node"]:
        """Pick the child with the highest UCB score."""
        return max(self.children.items(), key=lambda kv: kv[1].ucb_score(self.N))

    def most_visited_child(self) -> tuple[chess.Move, "Node"]:
        """After search is done, pick the move with the most visits (most reliable)."""
        return max(self.children.items(), key=lambda kv: kv[1].N)

    def visit_distribution(self, temperature: float = 1.0) -> dict[chess.Move, float]:
        """
        Turn visit counts into a probability distribution over moves.

        temperature=1: proportional to visits (used early in game — more variety)
        temperature→0: nearly deterministic, picks the most-visited move
        (temperature is applied as visit_count^(1/temp) before normalising)
        """
        if not self.children:
            return {}
        visits = {m: n.N for m, n in self.children.items()}
        if temperature == 0:
            best = max(visits, key=visits.get)
            return {m: (1.0 if m == best else 0.0) for m in visits}

        # Apply temperature
        powered = {m: v ** (1.0 / temperature) for m, v in visits.items()}
        total = sum(powered.values())
        return {m: v / total for m, v in powered.items()}


class MCTS:
    """
    The search engine. Owns the root node and runs simulations.

    Usage:
        mcts = MCTS(model, num_simulations=400)
        move = mcts.search(board)
    """

    def __init__(self, model, num_simulations: int = 200):
        self.model = model
        self.num_simulations = num_simulations

    def search(self, board: chess.Board, temperature: float = 1.0) -> chess.Move:
        """
        Run `num_simulations` simulations from `board`, return the chosen move.
        """
        root = Node(prior=1.0)
        self._expand(root, board)  # Expand root immediately so it has children

        for _ in range(self.num_simulations):
            node = root
            # stack=False: skip copying move history — MCTS never undoes moves, saves ~80× _BoardState allocs per sim
            scratch_board = board.copy(stack=False)
            path = [node]                 # Track nodes visited this simulation for backup

            # --- Step 1: Selection ---
            # Walk down the tree, always picking the highest-UCB child,
            # until we reach a node that hasn't been expanded yet.
            while node.is_expanded and not scratch_board.is_game_over():
                move, node = node.best_child()
                scratch_board.push(move)
                path.append(node)

            # --- Step 2: Expansion + Evaluation ---
            if scratch_board.is_game_over():
                # Terminal node: use the actual game result as the value
                result = scratch_board.result()
                value = self._terminal_value(result, scratch_board.turn)
            else:
                # Ask the network: what's the value here, and which moves look promising?
                value = self._expand(node, scratch_board)

            # --- Step 3: Backup ---
            # Send the value back up the path.
            # Flip sign at each level: what's good for the current player
            # is bad for the previous player (they're opponents).
            for i, visited_node in enumerate(reversed(path)):
                visited_node.N += 1
                # Flip value at each level to account for alternating players
                visited_node.W += value if i % 2 == 0 else -value

        # After all simulations: pick the move from the root's visit distribution
        dist = root.visit_distribution(temperature=temperature)
        moves = list(dist.keys())
        probs = [dist[m] for m in moves]
        chosen = np.random.choice(len(moves), p=probs)
        return moves[chosen]

    def get_policy(self, board: chess.Board, temperature: float = 1.0) -> dict[chess.Move, float]:
        """
        Run search and return the full visit distribution (used for training targets).
        This is the 'improved policy' that MCTS produces — better than the raw network output.
        """
        root = Node(prior=1.0)
        self._expand(root, board)

        for _ in range(self.num_simulations):
            node = root
            scratch_board = board.copy(stack=False)
            path = [node]

            while node.is_expanded and not scratch_board.is_game_over():
                move, node = node.best_child()
                scratch_board.push(move)
                path.append(node)

            if scratch_board.is_game_over():
                value = self._terminal_value(scratch_board.result(), scratch_board.turn)
            else:
                value = self._expand(node, scratch_board)

            for i, visited_node in enumerate(reversed(path)):
                visited_node.N += 1
                visited_node.W += value if i % 2 == 0 else -value

        return root.visit_distribution(temperature=temperature)

    def _expand(self, node: Node, board: chess.Board) -> float:
        """
        Run the neural network on this position.
        Populate the node's children with prior probabilities from the policy head.
        Returns the value estimate from the value head.
        """
        device = next(self.model.parameters()).device
        tensor = torch.from_numpy(board_to_tensor(board)).unsqueeze(0).float().to(device)
        with torch.no_grad():
            policy_logits, value = self.model(tensor)

        # Get legal move indices and mask out illegal moves
        legal_indices = get_legal_move_indices(board)
        policy = policy_logits.squeeze(0).cpu().numpy()

        # Softmax over legal moves only (illegal = -inf)
        legal_logits = np.array([policy[i] for i in legal_indices])
        legal_logits -= legal_logits.max()  # Numerical stability
        priors = np.exp(legal_logits)
        priors /= priors.sum()

        # Create a child node for each legal move with its prior probability
        for idx, move_idx in enumerate(legal_indices):
            move = index_to_move(move_idx, board)
            if move in board.legal_moves:
                node.children[move] = Node(prior=float(priors[idx]))

        node.is_expanded = True
        return float(value.cpu().item())

    def _terminal_value(self, result: str, turn: bool) -> float:
        """Convert PGN result string to a value from the current player's view."""
        if result == "1-0":
            return 1.0 if turn == chess.WHITE else -1.0
        elif result == "0-1":
            return -1.0 if turn == chess.WHITE else 1.0
        return 0.0  # Draw


Overwriting /content/chess-nn/chess_nn/mcts.py


: 

In [37]:
%%writefile /content/chess-nn/chess_nn/self_play.py
"""
Self-play: the model plays games against itself using MCTS.

For every position in every game we record:
  - board tensor          (input to the network)
  - MCTS visit distribution  (better policy target than raw network output)
  - game result           (value target: +1 win, 0 draw, -1 loss for the player to move)

Why is the MCTS distribution a better policy target than the raw network?
  The network makes one guess. MCTS runs hundreds of simulations and aggregates them.
  Training the network to imitate MCTS = the network learns to think ahead,
  even though it's just a single forward pass at inference time.

This file generates the data. train_rl.py consumes it.
"""

import gc
import os
import sys
import chess
import numpy as np
from tqdm import tqdm

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from config import PROCESSED_DATA_DIR, RL_CHUNK_SIZE
from chess_nn.board_encoding import board_to_tensor
from chess_nn.move_encoding import move_to_index
from chess_nn.mcts import MCTS

# Temperature schedule:
# Early in the game (< TEMP_THRESHOLD moves), use temperature=1 for variety.
# Late in the game, use near-deterministic play (temperature→0).
# This mirrors AlphaZero: explore early, exploit late.
TEMP_THRESHOLD = 30


def play_game(mcts: MCTS, max_moves: int = 200) -> list[dict]:
    """
    Play one full game of the model against itself using MCTS.

    Returns a list of position records — one per move played.
    Each record has board_tensor, policy (4672-length array), and turn (chess.WHITE/BLACK).
    The value target is filled in at the end once we know the game result.
    """
    board = chess.Board()
    positions = []

    for move_num in range(max_moves):
        if board.is_game_over():
            break

        temperature = 1.0 if move_num < TEMP_THRESHOLD else 0.1

        # Get MCTS visit distribution — this is the improved policy target
        visit_dist = mcts.get_policy(board, temperature=temperature)

        # Convert visit distribution to a 4672-length array
        policy_target = np.zeros(4672, dtype=np.float32)
        for move, prob in visit_dist.items():
            if move in board.legal_moves:
                idx = move_to_index(move, board)
                policy_target[idx] = prob

        positions.append({
            "board_tensor": board_to_tensor(board),
            "policy": policy_target,
            "turn": board.turn,
        })

        # Pick the move from the distribution
        moves = list(visit_dist.keys())
        probs = np.array([visit_dist[m] for m in moves])
        probs /= probs.sum()  # Re-normalise for safety
        chosen_move = np.random.choice(moves, p=probs)
        board.push(chosen_move)

    # --- Fill in value targets ---
    # Now we know who won. Go back through every position and assign
    # the game result from that player's perspective.
    result = board.result()
    records = []
    for pos in positions:
        if result == "1-0":
            value = 1.0 if pos["turn"] == chess.WHITE else -1.0
        elif result == "0-1":
            value = -1.0 if pos["turn"] == chess.WHITE else 1.0
        else:
            value = 0.0  # Draw

        records.append({
            "board_tensor": pos["board_tensor"],
            "policy": pos["policy"],
            "value": np.float32(value),
        })

    return records, result


def generate_games(model, num_games: int, num_simulations: int = 200,
                   output_dir: str = None, iteration: int = 0,
                   chunk_size: int = RL_CHUNK_SIZE) -> str:
    """
    Generate `num_games` self-play games and save them as a .npz file.

    Games are accumulated in RAM in batches of `chunk_size`, then flushed to temporary
    files on disk. This keeps peak RAM proportional to `chunk_size` × avg_game_length
    rather than `num_games` × avg_game_length.

    Returns the path to the final merged .npz file.
    """
    if output_dir is None:
        output_dir = os.path.join(PROCESSED_DATA_DIR, "self_play")
    os.makedirs(output_dir, exist_ok=True)

    mcts = MCTS(model, num_simulations=num_simulations)
    results = {"1-0": 0, "0-1": 0, "1/2-1/2": 0, "*": 0}
    total_positions = 0

    print(f"Generating {num_games} self-play games ({num_simulations} sims/move, "
          f"flushing every {chunk_size})...")

    chunk_paths = []
    all_boards, all_policies, all_values = [], [], []

    for game_idx in tqdm(range(num_games), desc="Self-play games"):
        records, result = play_game(mcts)
        results[result] = results.get(result, 0) + 1

        for r in records:
            all_boards.append(r["board_tensor"])
            all_policies.append(r["policy"])
            all_values.append(r["value"])
            total_positions += 1

        # Flush to disk every chunk_size games to keep RAM bounded
        is_last = (game_idx == num_games - 1)
        if (game_idx + 1) % chunk_size == 0 or is_last:
            chunk_path = os.path.join(
                output_dir, f"_tmp_iter{iteration:03d}_{len(chunk_paths):02d}.npz"
            )
            np.savez_compressed(
                chunk_path,
                boards=np.array(all_boards, dtype=np.float32),
                policies=np.array(all_policies, dtype=np.float32),
                values=np.array(all_values, dtype=np.float32),
            )
            chunk_paths.append(chunk_path)
            all_boards.clear()
            all_policies.clear()
            all_values.clear()
            gc.collect()

    # Merge all chunks into one file for this iteration
    final_path = os.path.join(output_dir, f"selfplay_iter{iteration:03d}.npz")
    if len(chunk_paths) == 1:
        os.rename(chunk_paths[0], final_path)
    else:
        merged = [np.load(p) for p in chunk_paths]
        np.savez_compressed(
            final_path,
            boards=np.concatenate([d["boards"] for d in merged]),
            policies=np.concatenate([d["policies"] for d in merged]),
            values=np.concatenate([d["values"] for d in merged]),
        )
        for p in chunk_paths:
            os.remove(p)

    total = num_games
    print(f"\nResults — White: {results['1-0']}/{total}  "
          f"Black: {results['0-1']}/{total}  "
          f"Draws: {results['1/2-1/2']}/{total}")
    print(f"Positions: {total_positions:,} → {final_path}")
    return final_path


Overwriting /content/chess-nn/chess_nn/self_play.py


: 

In [38]:
%%writefile /content/chess-nn/chess_nn/train_rl.py
"""
Reinforcement learning training loop — AlphaZero style.

Each iteration:
  1. Generate self-play games using the current model + MCTS
  2. Train on those games (policy loss + value loss)
  3. Evaluate new model vs old model (play N head-to-head games)
  4. Keep the winner, discard the loser
  5. Repeat

Why do we evaluate and potentially discard?
  Neural networks can occasionally get worse after an update if the self-play data
  was unlucky or the learning rate was too high. Head-to-head evaluation catches this.
  If the new model wins >55% of games, it's genuinely better. Otherwise keep the old one.

With 200 simulations and a small model, each iteration takes ~10-20 minutes on M-chip.
The model will gradually improve with each iteration.
"""

import gc
import os
import sys
import glob
import copy
import random
import torch
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from config import (
    CHECKPOINT_DIR, BATCH_SIZE, LEARNING_RATE, WEIGHT_DECAY, VALUE_LOSS_WEIGHT, DEVICE,
    RL_GAMES_PER_ITER, RL_SIMULATIONS, RL_CHUNK_SIZE,
    RL_HISTORY_FILES, RL_EPOCHS, RL_LR, RL_EVAL_GAMES, RL_WIN_THRESHOLD,
)
from chess_nn.model import ChessNet
from chess_nn.utils import save_checkpoint, load_checkpoint
from chess_nn.self_play import generate_games
from chess_nn.mcts import MCTS
import bisect
import chess


class SelfPlayDataset(Dataset):
    """
    Dataset for self-play data. Keeps each file's arrays separate instead of
    concatenating them — halves peak RAM vs np.concatenate (no second copy during merge).

    Policy targets are MCTS visit distributions (shape 4672), not single move indices.
    """

    def __init__(self, npz_paths: list[str], max_files: int = RL_HISTORY_FILES):
        paths = sorted(npz_paths)[-max_files:]
        self._boards: list[np.ndarray] = []
        self._policies: list[np.ndarray] = []
        self._values: list[np.ndarray] = []
        self._cumulative = [0]

        for path in paths:
            data = np.load(path)
            self._boards.append(data["boards"])
            self._policies.append(data["policies"])
            self._values.append(data["values"])
            self._cumulative.append(self._cumulative[-1] + len(data["boards"]))

        print(f"RL dataset: {self._cumulative[-1]:,} positions from {len(paths)} file(s)")

    def __len__(self):
        return self._cumulative[-1]

    def __getitem__(self, idx):
        # bisect finds which file contains this global index
        file_idx = bisect.bisect_right(self._cumulative, idx) - 1
        local_idx = idx - self._cumulative[file_idx]
        return (
            torch.from_numpy(self._boards[file_idx][local_idx].copy()),
            torch.from_numpy(self._policies[file_idx][local_idx].copy()),
            torch.tensor(self._values[file_idx][local_idx]),
        )


def train_on_selfplay(model, dataset: SelfPlayDataset, epochs: int = RL_EPOCHS, lr: float = RL_LR) -> float:
    """
    Train the model on self-play data for a few epochs.
    Returns the final average loss.

    Note: policy loss here uses KL divergence (not cross-entropy) because
    the target is a probability distribution from MCTS, not a single move index.
    KL divergence measures how different two probability distributions are.
    """
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)

    model.to(DEVICE)
    model.train()
    final_loss = 0.0

    for epoch in range(epochs):
        epoch_loss = 0.0
        for boards, policy_targets, value_targets in loader:
            boards = boards.to(DEVICE)
            policy_targets = policy_targets.to(DEVICE)
            value_targets = value_targets.to(DEVICE)
            policy_logits, value_pred = model(boards)

            # Policy loss: KL divergence between MCTS distribution and network output
            # log_softmax gives log-probabilities from the network
            # MCTS distribution is already probabilities (sums to 1)
            log_probs = F.log_softmax(policy_logits, dim=1)
            policy_loss = -(policy_targets * log_probs).sum(dim=1).mean()

            # Value loss: MSE between predicted win probability and actual outcome
            value_loss = F.mse_loss(value_pred.squeeze(1), value_targets)

            loss = policy_loss + VALUE_LOSS_WEIGHT * value_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_loss += loss.item()

        avg = epoch_loss / len(loader)
        print(f"  RL epoch {epoch+1}/{epochs}: loss={avg:.4f}")
        final_loss = avg

    return final_loss


def evaluate_models(new_model, old_model, num_games: int = RL_EVAL_GAMES,
                    simulations: int = 50) -> float:
    """
    Play head-to-head games between new and old model.
    Uses fewer simulations than training (50 vs 200) for speed.
    Returns new model win rate.

    Each model plays half the games as White, half as Black (to eliminate colour bias).
    """
    new_mcts = MCTS(new_model, num_simulations=simulations)
    old_mcts = MCTS(old_model, num_simulations=simulations)

    new_wins = draws = old_wins = 0

    for game_idx in range(num_games):
        board = chess.Board()
        new_is_white = (game_idx % 2 == 0)

        while not board.is_game_over():
            if (board.turn == chess.WHITE) == new_is_white:
                move = new_mcts.search(board, temperature=0)
            else:
                move = old_mcts.search(board, temperature=0)
            if move in board.legal_moves:
                board.push(move)
            else:
                board.push(random.choice(list(board.legal_moves)))

        result = board.result()
        if result == "1-0":
            if new_is_white:
                new_wins += 1
            else:
                old_wins += 1
        elif result == "0-1":
            if new_is_white:
                old_wins += 1
            else:
                new_wins += 1
        else:
            draws += 1

    del new_mcts, old_mcts

    win_rate = new_wins / num_games
    print(f"  Evaluation: new={new_wins} draws={draws} old={old_wins}  win_rate={win_rate:.2f}")
    return win_rate


def run_rl_loop(num_iterations: int = 10, start_checkpoint: str = "best_model.pt",
                games_per_iter: int = None, num_simulations: int = None,
                eval_games: int = None, rl_epochs: int = None):
    """
    Main RL loop. Runs `num_iterations` cycles of self-play → train → evaluate.

    Optional overrides (default to config.py values if not set):
      games_per_iter    — self-play games per iteration
      num_simulations   — MCTS simulations per move
      eval_games        — head-to-head games for model comparison
      rl_epochs         — training epochs per iteration
    """
    games_per_iter  = games_per_iter  or RL_GAMES_PER_ITER
    num_simulations = num_simulations or RL_SIMULATIONS
    eval_games      = eval_games      or RL_EVAL_GAMES
    rl_epochs       = rl_epochs       or RL_EPOCHS
    selfplay_dir = os.path.join(
        os.path.dirname(os.path.dirname(os.path.abspath(__file__))),
        "data", "processed", "self_play"
    )
    os.makedirs(selfplay_dir, exist_ok=True)

    # Load current best model
    model = ChessNet()
    checkpoint_path = os.path.join(CHECKPOINT_DIR, start_checkpoint)
    if os.path.exists(checkpoint_path):
        load_checkpoint(checkpoint_path, model)
        print(f"Starting from checkpoint: {checkpoint_path}")
    else:
        print("No checkpoint found — starting from scratch (random weights)")
    model.eval()

    for iteration in range(num_iterations):
        print(f"\n{'='*50}")
        print(f"ITERATION {iteration + 1} / {num_iterations}")
        print(f"{'='*50}")

        # Step 1: Generate self-play data — keep model on CPU so MCTS tensor ops stay cheap
        print("\n[1] Generating self-play games...")
        model.cpu()
        selfplay_path = generate_games(
            model,
            num_games=games_per_iter,
            num_simulations=num_simulations,
            output_dir=selfplay_dir,
            iteration=iteration,
        )
        gc.collect()

        # Step 2: Train a copy of the model on new self-play data (uses DEVICE / MPS)
        print("\n[2] Training on self-play data...")
        new_model = copy.deepcopy(model)

        all_selfplay = sorted(glob.glob(os.path.join(selfplay_dir, "selfplay_iter*.npz")))
        dataset = SelfPlayDataset(all_selfplay)
        train_on_selfplay(new_model, dataset, epochs=rl_epochs)  # moves new_model to DEVICE internally
        del dataset
        new_model.cpu()
        new_model.eval()
        gc.collect()
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            torch.mps.empty_cache()

        # Step 3: Evaluate new model vs old (both on CPU for MCTS)
        print("\n[3] Evaluating new model vs old...")
        old_model = copy.deepcopy(model)
        old_model.eval()
        win_rate = evaluate_models(new_model, old_model, num_games=eval_games)
        del old_model
        gc.collect()

        # Step 4: Keep winner
        if win_rate >= RL_WIN_THRESHOLD:
            print(f"\n  New model promoted (win rate {win_rate:.0%} >= {RL_WIN_THRESHOLD:.0%})")
            del model
            model = new_model
            save_checkpoint(model, torch.optim.Adam(model.parameters()), iteration, win_rate,
                            f"rl_best_model.pt")
            save_checkpoint(model, torch.optim.Adam(model.parameters()), iteration, win_rate,
                            f"rl_iter_{iteration:03d}.pt")
        else:
            print(f"\n  Old model kept (new win rate {win_rate:.0%} < {RL_WIN_THRESHOLD:.0%})")
            del new_model
        gc.collect()

    print("\nRL training complete.")
    return model


if __name__ == "__main__":
    run_rl_loop(num_iterations=5)


Overwriting /content/chess-nn/chess_nn/train_rl.py


: 

In [39]:
%%writefile /content/chess-nn/data/__init__.py


Overwriting /content/chess-nn/data/__init__.py


: 

In [40]:
%%writefile /content/chess-nn/data/download_data.py
"""
Download and filter games from the Lichess open database.

Lichess publishes every rated game ever played, freely available at:
  https://database.lichess.org/

Files are compressed with zstandard (.zst). We stream-decompress on the fly
so we never need to store the full uncompressed file (they're huge — GB range).

We filter for quality: both players 2000+ rated, standard rules, 10+ moves.
Target: ~100,000 games → ~3-5 million board positions for training.
"""

import os
import sys
import requests
import zstandard as zstd
import chess.pgn
import io
from tqdm import tqdm

# Add parent dir so we can import config
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import RAW_DATA_DIR, MIN_RATING, MIN_MOVES

# Lichess database URL — January 2024 standard rated games
# Change the date suffix to get a different month
LICHESS_URL = "https://database.lichess.org/standard/lichess_db_standard_rated_2024-01.pgn.zst"
OUTPUT_FILE = os.path.join(RAW_DATA_DIR, "filtered_games.pgn")
TARGET_GAMES = 50_000


def _clear_old_data():
    import shutil
    from config import PROCESSED_DATA_DIR

    cleared = []

    # Raw PGN
    if os.path.exists(OUTPUT_FILE):
        os.remove(OUTPUT_FILE)
        cleared.append(OUTPUT_FILE)

    # Processed .npz chunks
    if os.path.exists(PROCESSED_DATA_DIR):
        shutil.rmtree(PROCESSED_DATA_DIR)
        cleared.append(PROCESSED_DATA_DIR)

    if cleared:
        print("Cleared old data:")
        for p in cleared:
            print(f"  {p}")
        print()


def download_and_filter():
    _clear_old_data()
    os.makedirs(RAW_DATA_DIR, exist_ok=True)

    print(f"Downloading from: {LICHESS_URL}")
    print(f"Target: {TARGET_GAMES:,} games with both players rated {MIN_RATING}+")
    print("This streams the file — no need to download the full archive first.\n")

    response = requests.get(LICHESS_URL, stream=True)
    response.raise_for_status()

    # zstd streaming decompressor
    dctx = zstd.ZstdDecompressor()
    stream_reader = dctx.stream_reader(response.raw)  # type: ignore[arg-type]
    text_stream = io.TextIOWrapper(stream_reader, encoding="utf-8", errors="replace")

    games_written = 0
    games_seen = 0

    with open(OUTPUT_FILE, "w") as out_file:
        pbar = tqdm(total=TARGET_GAMES, desc="Games collected", unit="game")

        while games_written < TARGET_GAMES:
            game = chess.pgn.read_game(text_stream)
            if game is None:
                print("\nReached end of database file.")
                break

            games_seen += 1

            # Filter: ratings
            white_elo = int(game.headers.get("WhiteElo", "0") or "0")
            black_elo = int(game.headers.get("BlackElo", "0") or "0")
            if white_elo < MIN_RATING or black_elo < MIN_RATING:
                continue

            # Filter: game must have a result (not abandoned)
            result = game.headers.get("Result", "*")
            if result not in ("1-0", "0-1", "1/2-1/2"):
                continue

            # Filter: enough moves
            moves = list(game.mainline_moves())
            if len(moves) < MIN_MOVES:
                continue

            # Write to output PGN
            print(game, file=out_file, end="\n\n")
            games_written += 1
            pbar.update(1)

        pbar.close()

    print(f"\nDone. Seen {games_seen:,} games, kept {games_written:,}.")
    print(f"Saved to: {OUTPUT_FILE}")
    return OUTPUT_FILE


if __name__ == "__main__":
    download_and_filter()


Overwriting /content/chess-nn/data/download_data.py


: 

In [41]:
import sys
sys.path.insert(0, '/content/chess-nn')
from chess_nn.model import ChessNet
m = ChessNet()
params = sum(p.numel() for p in m.parameters())
print(f'OK — {params:,} parameters')


OK — 2,118,087 parameters


: 

## 3. Download + Process Data

Streams ~20k rated games from Lichess and converts to `.npz` chunks. ~10-20 min.


In [42]:
import sys; sys.path.insert(0, '/content/chess-nn')
from data.download_data import download_and_filter
download_and_filter()


Cleared old data:
  /content/chess-nn/data/raw/filtered_games.pgn
  /content/chess-nn/data/processed

Target: 20,000 games with both players rated 2000+
This streams the file — no need to download the full archive first.



Games collected: 100%|██████████| 20000/20000 [05:41<00:00, 58.61game/s]


Done. Seen 116,322 games, kept 20,000.
Saved to: /content/chess-nn/data/raw/filtered_games.pgn


'/content/chess-nn/data/raw/filtered_games.pgn'

: 

In [43]:
import glob
from chess_nn.dataset import process_games
for pgn in sorted(glob.glob('/content/chess-nn/data/raw/*.pgn')):
    process_games(pgn)


Processing games from: /content/chess-nn/data/raw/filtered_games.pgn


Processing: 100245pos [01:31, 110.27pos/s, chunk=1, games=1304] 


Saved chunk 0 (100,000 positions) → /content/chess-nn/data/processed/chunk_0000.npz


Processing: 200241pos [03:01, 102.91pos/s, chunk=2, games=2590] 


Saved chunk 1 (100,000 positions) → /content/chess-nn/data/processed/chunk_0001.npz


...

Processing: 1500239pos [22:43, 108.46pos/s, chunk=15, games=19681] 


Saved chunk 14 (100,000 positions) → /content/chess-nn/data/processed/chunk_0014.npz


Processing: 1524789pos [23:04, 1101.42pos/s, chunk=15, games=2e+4] 



Saved chunk 15 (24,789 positions) → /content/chess-nn/data/processed/chunk_0015.npz
Total positions: 1,524,789 across 16 chunks


: 

## 4. Supervised Training

15 epochs on Lichess positions. ~5-10 min/epoch on T4 GPU (~2 hrs total).
To resume after interruption: change `resume=False` to `resume=True`.


In [ ]:
import glob
from chess_nn.train import train
chunk_paths = sorted(glob.glob('/content/chess-nn/data/processed/chunk_*.npz'))
print(f'{len(chunk_paths)} chunks')
train(chunk_paths, resume=False)


16 chunks
Training on device: cuda
Dataset: 1,524,789 positions across 16 chunks (lazy)
Train: 1,372,310 | Val: 76,239 | Test: 76,240

  Epoch 1 / 15
  Note: first ~20 batches are slow while MPS warms up.
    536/5369  [ 10.0%]  policy 3.121  value 0.901  lr 0.00053  |  1m 22s elapsed  ETA 12m 25s
   1072/5369  [ 20.0%]  policy 3.077  value 0.865  lr 0.00100  |  2m 44s elapsed  ETA 10m 58s
   1608/5369  [ 29.9%]  policy 3.040  value 0.849  lr 0.00100  |  4m 04s elapsed  ETA 9m 31s
   2144/5369  [ 39.9%]  policy 3.011  value 0.831  lr 0.00100  |  5m 25s elapsed  ETA 8m 08s
   2680/5369  [ 49.9%]  policy 2.989  value 0.821  lr 0.00100  |  6m 46s elapsed  ETA 6m 47s
   3216/5369  [ 59.9%]  policy 2.972  value 0.810  lr 0.00100  |  8m 06s elapsed  ETA 5m 25s
   3752/5369  [ 69.9%]  policy 2.958  value 0.806  lr 0.00100  |  9m 27s elapsed  ETA 4m 04s
   4288/5369  [ 79.9%]  policy 2.945  value 0.796  lr 0.00100  |  10m 47s elapsed  ETA 2m 43s
   4824/5369  [ 89.8%]  policy 2.935  value 0.79

## 5. RL Self-Play Training

10 AlphaZero-style iterations. ~30-60 min each on T4.


In [ ]:
from chess_nn.train_rl import run_rl_loop
run_rl_loop(num_iterations=10, start_checkpoint='best_model.pt',
            games_per_iter=25, num_simulations=200)


: 

## 6. Download Results

In [ ]:
import os, glob
ckpts = sorted(glob.glob('/content/chess-nn/checkpoints_new/*.pt'))
print(f'{len(ckpts)} checkpoints:')
for p in ckpts:
    print(f'  {os.path.basename(p):<35} {os.path.getsize(p)/1e6:.1f} MB')


: 

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/chess_nn_checkpoints', 'zip',
                    '/content/chess-nn/checkpoints_new')
files.download('/content/chess_nn_checkpoints.zip')


: 